# Day 3 · 极简 softmax 手算：3 个词的自注意力权重矩阵

**目标**：亲手算出一个 3×3 的注意力权重矩阵，每行和为 1，长得像一张**转移概率矩阵**。

**三步走**（自注意力的全部核心）：
1. Q 与所有 K 做点积 → 相似度分数
2. softmax 归一化 → 权重矩阵（每行和为 1）
3. 权重 × V 加权平均 → 融合上下文的新词向量

> 运行方法：依次按 `Shift + Enter` 运行每个格子。

In [1]:
import numpy as np

# 3 个词的词向量（极简 one-hot，便于手算）
words = ["猫", "追", "狗"]
X = np.eye(3)   # 猫=[1,0,0] 追=[0,1,0] 狗=[0,0,1]

## 第 1 步：生成 Q / K / V

真实模型里变换矩阵 $W_q, W_k, W_v$ 是训练学出来的；这里先用单位矩阵，让 Q=K=V=词向量，排除干扰、只看流程。

In [ ]:
Wq = Wk = Wv = np.eye(3)
# @ 矩阵乘法
Q = X @ Wq   # Query
K = X @ Wk   # Key
V = X @ Wv   # Value

print("每个词的 Q（这里 Q=K=V）：")
for i, w in enumerate(words):
    print(f"{w}: Q={Q[i]}")

# 相似度分数矩阵：Q 与所有 K 做点积
scores = Q @ K.T
# .T 代表转置
print("\n相似度分数矩阵 scores = Q·K^T：")
print(np.round(scores, 4))

每个词的 Q（这里 Q=K=V）：
猫: Q=[1. 0. 0.]
追: Q=[0. 1. 0.]
狗: Q=[0. 0. 1.]

相似度分数矩阵 scores = Q·K^T：
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


## 第 2 步：softmax 把每行分数变成概率分布

softmax 输出 = **概率分布**：每格 0~1，每行和为 1。

> 数值技巧：先减去行最大值再取 exp，防止指数溢出（对结果无影响）。

In [3]:
def softmax(row):
    row = row - row.max()   # 数值稳定
    e = np.exp(row)
    return e / e.sum()

weights = np.array([softmax(r) for r in scores])
print("softmax 后的权重矩阵（每行和为 1）：")
print(np.round(weights, 4))
print("每行之和:", np.round(weights.sum(axis=1), 4))

softmax 后的权重矩阵（每行和为 1）：
[[0.5761 0.2119 0.2119]
 [0.2119 0.5761 0.2119]
 [0.2119 0.2119 0.5761]]
每行之和: [1. 1. 1.]


## 第 3 步：权重 × V 加权平均 → 融合上下文的新词向量

这一步就是 Day 2 那句"**注意力 = 对 Value 的加权平均**"的落地。

In [4]:
out = weights @ V
print("融合上下文后的新词向量：")
for i, w in enumerate(words):
    print(f"{w} 的新表示: {np.round(out[i], 4)}")

print("\n数学视角：权重矩阵每行和=1、每格 0~1 → 和'转移概率矩阵'长得一模一样。")
print("第 i 行 = 词 i 从各词收集信息的比例。")

融合上下文后的新词向量：
猫 的新表示: [0.5761 0.2119 0.2119]
追 的新表示: [0.2119 0.5761 0.2119]
狗 的新表示: [0.2119 0.2119 0.5761]

数学视角：权重矩阵每行和=1、每格 0~1 → 和'转移概率矩阵'长得一模一样。
第 i 行 = 词 i 从各词收集信息的比例。


## 动手改一改：给"猫""狗"加上共同的"动物"维度

观察：当"猫""狗"共享"动物"语义维度后，它们对彼此的权重是否明显变大？而"追"（没有该维度）是否被忽略？

In [ ]:
X2 = np.array([
    [1, 0, 1],   # 猫：有"动物"维度
    [0, 1, 0],   # 追：动作，没有"动物"维度
    [1, 0, 1],   # 狗：有"动物"维度
]).astype(float)

Q2, K2, V2 = X2, X2, X2
scores2 = Q2 @ K2.T
weights2 = np.array([softmax(r) for r in scores2])
print("升级后的权重矩阵（每行和为 1）：")
print(np.round(weights2, 4))
print("\n'猫''狗'（第 1、3 行）对彼此的权重变大：0.212 → 0.468；对'追'掉到 0.063。")
print("结论：词向量承载语义 → 注意力自动聚焦到同类词。")

## 小结（写进你的概念笔记）

1. 自注意力 = `Q·Kᵀ → softmax → ×V` 三步；
2. softmax 输出 = 概率分布；权重矩阵 = 转移概率矩阵；
3. 词向量承载语义 → 注意力自动发现词间关系（升级版实验验证）。

> 下一站：Day 4 多头注意力 = 多组 Q/K/V 并行，多个视角。